# SARSA — Implementación desde Cero

**SARSA** (State-Action-Reward-State-Action) es uno de los algoritmos de aprendizaje por refuerzo más clásicos y didácticos. A diferencia de DQN, no usa redes neuronales: aprende directamente con una tabla de valores Q.

En este notebook implementaremos SARSA **desde cero en Python puro** (solo numpy), aplicado al entorno `FrozenLake-v1` de Gymnasium.

---

## ¿Qué es FrozenLake?

Un grid de 4×4 donde el agente debe ir desde la casilla de inicio (S) hasta la meta (G) sin caer en los agujeros (H):

```
S F F F
F H F H
F F F H
H F F G
```

- **S** = Start (inicio)
- **F** = Frozen (suelo helado, seguro)
- **H** = Hole (agujero, episodio terminado con recompensa 0)
- **G** = Goal (meta, recompensa 1)

**Espacio de estados:** 16 casillas (0–15)  
**Espacio de acciones:** 4 direcciones (0=izquierda, 1=abajo, 2=derecha, 3=arriba)

---
## Paso 0 — Instalación de dependencias

In [ ]:
# Solo necesitamos gymnasium y numpy
# matplotlib es opcional, para visualizar los resultados
import sys
!{sys.executable} -m pip install gymnasium numpy matplotlib --quiet

---
## Paso 1 — Importaciones

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Para reproducibilidad
np.random.seed(42)

print(f'numpy   : {np.__version__}')
print(f'gymnasium: {gym.__version__}')

---
## Paso 2 — Crear el entorno

In [ ]:
# is_slippery=False hace el entorno determinista (más fácil para aprender).
# Con is_slippery=True el suelo es resbaladizo y las acciones pueden
# no ejecutarse como se espera, añadiendo estocasticidad.
env = gym.make('FrozenLake-v1', is_slippery=False, render_mode='ansi')

n_states  = env.observation_space.n   # 16 estados posibles
n_actions = env.action_space.n        # 4 acciones posibles

print(f'Número de estados : {n_states}')
print(f'Número de acciones: {n_actions}')
print()

# Visualizamos el mapa del entorno
obs, info = env.reset(seed=42)
print(env.render())

---
## Paso 3 — La Tabla Q

SARSA guarda lo aprendido en una **tabla Q** de dimensiones `[estados × acciones]`.

Cada celda `Q[s, a]` almacena la estimación de la recompensa futura acumulada que el agente espera obtener si está en el estado `s` y ejecuta la acción `a`.

Al inicio, todos los valores son 0 (el agente no sabe nada).

In [ ]:
# Inicializamos la tabla Q con ceros
# Forma: (16 estados, 4 acciones)
Q = np.zeros((n_states, n_actions))

print('Tabla Q inicial (todos los valores a 0):')
print(f'Forma: {Q.shape}  →  {n_states} estados × {n_actions} acciones')
print()
print('       Izq   Abj   Der   Arr')
print('       (0)   (1)   (2)   (3)')
for i, row in enumerate(Q):
    print(f'  s={i:2d}  {row}')

---
## Paso 4 — Hiperparámetros

| Parámetro | Símbolo | Descripción |
|-----------|---------|-------------|
| `alpha`   | α       | **Tasa de aprendizaje**: cuánto actualiza el agente sus estimaciones. 0 = no aprende, 1 = sobreescribe todo. |
| `gamma`   | γ       | **Factor de descuento**: importancia de recompensas futuras. 0 = solo presente, 1 = horizonte infinito. |
| `epsilon` | ε       | **Exploración**: probabilidad de tomar una acción aleatoria. |
| `epsilon_min` | — | Valor mínimo de epsilon (siempre hay algo de exploración). |
| `epsilon_decay` | — | Cuánto reduce epsilon en cada episodio. |
| `n_episodes` | — | Número total de episodios de entrenamiento. |

In [ ]:
alpha         = 0.8    # tasa de aprendizaje
gamma         = 0.95   # factor de descuento
epsilon       = 1.0    # exploración inicial (100% aleatoria)
epsilon_min   = 0.01   # mínimo de exploración
epsilon_decay = 0.995  # multiplicador de decaimiento por episodio
n_episodes    = 2000   # número de episodios de entrenamiento

print('Hiperparámetros configurados:')
print(f'  α (learning rate)  = {alpha}')
print(f'  γ (discount)       = {gamma}')
print(f'  ε inicial          = {epsilon}')
print(f'  ε mínimo           = {epsilon_min}')
print(f'  ε decaimiento      = {epsilon_decay}')
print(f'  Episodios          = {n_episodes}')

---
## Paso 5 — Política ε-greedy

La política **ε-greedy** decide cómo el agente elige acciones:

- Con probabilidad **ε** → acción **aleatoria** (exploración: descubrir cosas nuevas)
- Con probabilidad **1-ε** → acción con mayor valor Q (explotación: usar lo aprendido)

Al inicio ε=1 (el agente explora completamente). Con el tiempo ε decrece y el agente explota más su conocimiento.

In [ ]:
def elegir_accion(estado, Q, epsilon):
    """
    Política ε-greedy: con probabilidad epsilon elige una acción aleatoria,
    con probabilidad (1-epsilon) elige la acción con mayor valor Q.

    Parámetros:
        estado  : estado actual del agente (int)
        Q       : tabla Q (numpy array)
        epsilon : probabilidad de exploración (float)

    Retorna:
        acción elegida (int)
    """
    if np.random.random() < epsilon:
        # Exploración: acción aleatoria
        return env.action_space.sample()
    else:
        # Explotación: acción con mayor valor Q en el estado actual
        return np.argmax(Q[estado])


# Ejemplo de uso
estado_ejemplo = 0
print(f'Acción elegida en el estado {estado_ejemplo}: {elegir_accion(estado_ejemplo, Q, epsilon=1.0)} (epsilon=1.0, siempre aleatoria)')
print(f'Acción elegida en el estado {estado_ejemplo}: {elegir_accion(estado_ejemplo, Q, epsilon=0.0)} (epsilon=0.0, siempre greedy)')

---
## Paso 6 — La Actualización SARSA

El núcleo del algoritmo es la **regla de actualización de SARSA**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \left[ r + \gamma \cdot Q(s', a') - Q(s, a) \right]$$

Donde:
- `s`  = estado actual
- `a`  = acción tomada
- `r`  = recompensa recibida
- `s'` = estado siguiente
- `a'` = **acción siguiente** (la que el agente *realmente* tomará con su política ε-greedy)

### ¿En qué se diferencia SARSA de Q-Learning?

| | SARSA (On-Policy) | Q-Learning (Off-Policy) |
|-|-------------------|-----------------------|
| Actualización con | Q(s', **a'**) — la acción que SE TOMARÁ | max Q(s', **a**) — la acción ÓPTIMA |
| Tipo | On-policy (aprende de lo que hace) | Off-policy (aprende del óptimo) |
| Más conservador | ✅ Sí | ❌ No |

SARSA usa la acción **real** que tomará el agente (incluyendo la exploración), no la mejor posible.

In [ ]:
def actualizar_Q(Q, estado, accion, recompensa, estado_sig, accion_sig, alpha, gamma):
    """
    Aplica la regla de actualización SARSA:
    Q(s,a) ← Q(s,a) + α * [r + γ * Q(s',a') - Q(s,a)]

    Parámetros:
        Q          : tabla Q (numpy array, se modifica in-place)
        estado     : estado actual s
        accion     : acción tomada a
        recompensa : recompensa recibida r
        estado_sig : estado siguiente s'
        accion_sig : acción siguiente a' (elegida por ε-greedy)
        alpha      : tasa de aprendizaje α
        gamma      : factor de descuento γ
    """
    # Valor Q actual: lo que creemos que vale (s, a)
    q_actual = Q[estado, accion]

    # Valor Q del siguiente paso: recompensa inmediata + valor futuro descontado
    # Nota: usamos Q(s', a') con la acción que REALMENTE tomará el agente (a')
    q_siguiente = Q[estado_sig, accion_sig]

    # Error TD (Temporal Difference): diferencia entre lo estimado y la nueva info
    td_error = recompensa + gamma * q_siguiente - q_actual

    # Actualización: movemos Q(s,a) en la dirección del error TD
    Q[estado, accion] = q_actual + alpha * td_error


# Ejemplo conceptual para ver cómo funciona la actualización
Q_test = np.zeros((n_states, n_actions))
print('Antes de actualizar: Q[0, 2] =', Q_test[0, 2])
actualizar_Q(Q_test, estado=0, accion=2, recompensa=1.0,
             estado_sig=1, accion_sig=1, alpha=0.8, gamma=0.95)
print('Después de actualizar: Q[0, 2] =', Q_test[0, 2])
print('(El valor subió porque recibimos recompensa positiva)')

---
## Paso 7 — Bucle de Entrenamiento

El bucle de entrenamiento SARSA sigue este esquema para cada episodio:

```
1. Reiniciar el entorno → obtener estado inicial s
2. Elegir acción a con política ε-greedy
3. Mientras el episodio no termine:
     a. Ejecutar acción a → obtener (r, s')
     b. Elegir acción siguiente a' con política ε-greedy
     c. Actualizar Q(s,a) con la regla SARSA
     d. s ← s' , a ← a'
4. Reducir epsilon (menos exploración con el tiempo)
```

La diferencia clave con Q-Learning: el paso (b) ocurre **antes** de la actualización y usamos `a'` en la fórmula.

In [ ]:
# Reiniciamos la tabla Q para el entrenamiento real
Q = np.zeros((n_states, n_actions))
epsilon = 1.0  # reiniciamos epsilon también

# Listas para registrar métricas
recompensas_episodio = []   # recompensa total de cada episodio
epsilons             = []   # valor de epsilon en cada episodio
exitos               = []   # 1 si el episodio fue exitoso (llegó a la meta), 0 si no

print(f'Entrenando SARSA durante {n_episodes} episodios...')
print('-' * 50)

for episodio in range(n_episodes):

    # ── 1. Reiniciar el entorno ──────────────────────────
    estado, info = env.reset()
    recompensa_total = 0
    terminado = False

    # ── 2. Elegir la primera acción ──────────────────────
    accion = elegir_accion(estado, Q, epsilon)

    # ── 3. Bucle del episodio ────────────────────────────
    while not terminado:

        # a) Ejecutar la acción en el entorno
        estado_sig, recompensa, terminado, truncado, info = env.step(accion)
        recompensa_total += recompensa

        # b) Elegir la SIGUIENTE acción con ε-greedy
        #    (esto es lo que diferencia SARSA de Q-Learning)
        accion_sig = elegir_accion(estado_sig, Q, epsilon)

        # c) Actualizar la tabla Q con la regla SARSA
        actualizar_Q(Q, estado, accion, recompensa,
                     estado_sig, accion_sig, alpha, gamma)

        # d) Avanzar al siguiente paso
        estado = estado_sig
        accion = accion_sig

        # Si el episodio fue truncado (tiempo límite) también terminamos
        if truncado:
            terminado = True

    # ── 4. Reducir epsilon (menos exploración con el tiempo) ──
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Registrar métricas
    recompensas_episodio.append(recompensa_total)
    epsilons.append(epsilon)
    exitos.append(1 if recompensa_total > 0 else 0)

    # Mostrar progreso cada 500 episodios
    if (episodio + 1) % 500 == 0:
        tasa_exito = np.mean(exitos[-500:]) * 100
        print(f'Episodio {episodio+1:5d}/{n_episodes} | '
              f'ε = {epsilon:.4f} | '
              f'Tasa de éxito (últimos 500): {tasa_exito:.1f}%')

print('-' * 50)
print('Entrenamiento completado.')

---
## Paso 8 — La Tabla Q Aprendida

Veamos cómo ha quedado la tabla Q después del entrenamiento. Las celdas con valores más altos indican acciones más prometedoras en cada estado.

In [ ]:
nombres_acciones = ['←', '↓', '→', '↑']

print('Tabla Q aprendida:')
print(f'  Estado  |  ← (0)   ↓ (1)   → (2)   ↑ (3)  | Mejor acción')
print('-' * 60)

for s in range(n_states):
    vals = Q[s]
    mejor = np.argmax(vals)
    print(f'  s = {s:2d}  |  '
          + '  '.join(f'{v:6.3f}' for v in vals)
          + f'  |  {nombres_acciones[mejor]} ({mejor})')

---
## Paso 9 — Visualización del Entrenamiento

In [ ]:
# Suavizamos la curva de recompensas con una media móvil
def media_movil(datos, ventana=100):
    return np.convolve(datos, np.ones(ventana)/ventana, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Entrenamiento SARSA — FrozenLake-v1', fontsize=14, fontweight='bold')

# ── Gráfico 1: Recompensa por episodio ──
ax = axes[0]
ax.plot(recompensas_episodio, alpha=0.3, color='steelblue', linewidth=0.8, label='Recompensa')
ax.plot(range(99, n_episodes), media_movil(recompensas_episodio),
        color='steelblue', linewidth=2, label='Media móvil (100)')
ax.set_title('Recompensa por episodio')
ax.set_xlabel('Episodio')
ax.set_ylabel('Recompensa total')
ax.legend()
ax.grid(alpha=0.3)

# ── Gráfico 2: Tasa de éxito ──
ax = axes[1]
tasa_exito_movil = media_movil(exitos, ventana=100) * 100
ax.plot(range(99, n_episodes), tasa_exito_movil, color='seagreen', linewidth=2)
ax.axhline(y=tasa_exito_movil[-1], color='red', linestyle='--', alpha=0.7,
           label=f'Final: {tasa_exito_movil[-1]:.1f}%')
ax.set_title('Tasa de éxito (media móvil 100 ep.)')
ax.set_xlabel('Episodio')
ax.set_ylabel('Éxito (%)')
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.3)

# ── Gráfico 3: Decaimiento de epsilon ──
ax = axes[2]
ax.plot(epsilons, color='darkorange', linewidth=2)
ax.set_title('Decaimiento de ε (exploración)')
ax.set_xlabel('Episodio')
ax.set_ylabel('ε (epsilon)')
ax.fill_between(range(n_episodes), epsilons, alpha=0.2, color='darkorange')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Paso 10 — Visualizar la Política Aprendida

Mostramos en el grid 4×4 qué acción recomienda el agente en cada casilla.

In [ ]:
# Mapa del entorno FrozenLake
mapa = [
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G'],
]

# Política: mejor acción en cada estado
politica = [np.argmax(Q[s]) for s in range(n_states)]
flechas  = ['←', '↓', '→', '↑']

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Política aprendida por SARSA', fontsize=14, fontweight='bold', pad=15)

colores = {'S': '#AED6F1', 'F': '#EAF2FF', 'H': '#E74C3C', 'G': '#2ECC71'}

for fila in range(4):
    for col in range(4):
        estado = fila * 4 + col
        tipo   = mapa[fila][col]
        x = col
        y = 3 - fila  # invertimos el eje y para que el origen quede arriba

        # Dibujar celda
        rect = mpatches.FancyBboxPatch(
            (x + 0.05, y + 0.05), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            facecolor=colores[tipo], edgecolor='gray', linewidth=1.5
        )
        ax.add_patch(rect)

        # Etiqueta del tipo de casilla
        ax.text(x + 0.5, y + 0.75, tipo, ha='center', va='center',
                fontsize=11, fontweight='bold', color='#2C3E50')

        # Número de estado
        ax.text(x + 0.15, y + 0.2, str(estado), ha='center', va='center',
                fontsize=8, color='gray')

        # Flecha de la política (solo en casillas transitables)
        if tipo not in ('H', 'G'):
            accion = politica[estado]
            ax.text(x + 0.5, y + 0.38, flechas[accion], ha='center', va='center',
                    fontsize=22, color='#2C3E50')

# Leyenda
leyenda = [
    mpatches.Patch(color='#AED6F1', label='S = Inicio'),
    mpatches.Patch(color='#EAF2FF', label='F = Suelo seguro'),
    mpatches.Patch(color='#E74C3C', label='H = Agujero'),
    mpatches.Patch(color='#2ECC71', label='G = Meta'),
]
ax.legend(handles=leyenda, loc='upper right', bbox_to_anchor=(1.35, 1.0), fontsize=10)

plt.tight_layout()
plt.show()

print('\nPolítica en texto:')
for fila in range(4):
    linea = ''
    for col in range(4):
        estado = fila * 4 + col
        tipo = mapa[fila][col]
        if tipo in ('H', 'G'):
            linea += f'  {tipo} '
        else:
            linea += f'  {flechas[politica[estado]]} '
    print(linea)

---
## Paso 11 — Evaluar el Agente Entrenado

Ejecutamos el agente sin exploración (epsilon=0) para medir su rendimiento real.

In [ ]:
def evaluar_agente(Q, env, n_episodios=100, verbose=False):
    """
    Evalúa el agente usando la política greedy pura (sin exploración).
    epsilon=0 significa que siempre elige la acción con mayor Q.
    """
    exitos = 0
    recompensas = []

    for ep in range(n_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        terminado = False
        pasos = []

        while not terminado:
            # epsilon=0: siempre la acción óptima aprendida
            accion = np.argmax(Q[estado])
            pasos.append((estado, flechas[accion]))

            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa

            if truncado:
                terminado = True

        recompensas.append(recompensa_total)
        if recompensa_total > 0:
            exitos += 1

        if verbose and ep < 3:
            print(f'Episodio {ep+1}: ', end='')
            for s, a in pasos:
                print(f's{s}{a}', end=' → ')
            print(f's{estado} | Recompensa: {recompensa_total}')

    tasa = exitos / n_episodios * 100
    return tasa, np.mean(recompensas)


tasa, recompensa_media = evaluar_agente(Q, env, n_episodios=100, verbose=True)

print()
print('=' * 40)
print(f'  Tasa de éxito     : {tasa:.1f}%')
print(f'  Recompensa media  : {recompensa_media:.3f}')
print('=' * 40)

---
## Paso 12 — Comparativa con Q-Learning

Para entender mejor SARSA, comparamos su curva de aprendizaje con Q-Learning.
La única diferencia está en la línea de actualización:

- **SARSA:** `Q(s,a) += α * [r + γ * Q(s', a') - Q(s,a)]`  ← usa la acción real `a'`
- **Q-Learning:** `Q(s,a) += α * [r + γ * max Q(s', *) - Q(s,a)]`  ← usa el máximo

In [ ]:
def entrenar_qlearning(n_episodes, alpha, gamma, epsilon_ini, epsilon_min, epsilon_decay):
    """Entrena Q-Learning. Idéntico a SARSA salvo la línea de actualización."""
    env_ql = gym.make('FrozenLake-v1', is_slippery=False)
    Q_ql = np.zeros((n_states, n_actions))
    eps = epsilon_ini
    exitos = []

    for _ in range(n_episodes):
        estado, _ = env_ql.reset()
        terminado = False
        recompensa_total = 0

        while not terminado:
            accion = elegir_accion(estado, Q_ql, eps)
            estado_sig, recompensa, terminado, truncado, _ = env_ql.step(accion)
            recompensa_total += recompensa

            # ← DIFERENCIA: usamos max Q(s', *) en lugar de Q(s', a')
            q_actual   = Q_ql[estado, accion]
            q_max_sig  = np.max(Q_ql[estado_sig])       # <-- aquí está la diferencia
            td_error   = recompensa + gamma * q_max_sig - q_actual
            Q_ql[estado, accion] += alpha * td_error

            estado = estado_sig
            if truncado:
                terminado = True

        eps = max(epsilon_min, eps * epsilon_decay)
        exitos.append(1 if recompensa_total > 0 else 0)

    env_ql.close()
    return exitos


# Entrenamos Q-Learning con los mismos hiperparámetros
exitos_ql = entrenar_qlearning(n_episodes, alpha, gamma, 1.0, epsilon_min, epsilon_decay)

# Comparamos curvas de éxito
ventana = 100
sarsa_smooth = media_movil(exitos, ventana) * 100
ql_smooth    = media_movil(exitos_ql, ventana) * 100

plt.figure(figsize=(10, 5))
plt.plot(range(ventana-1, n_episodes), sarsa_smooth, label='SARSA (on-policy)', color='steelblue', linewidth=2)
plt.plot(range(ventana-1, n_episodes), ql_smooth,    label='Q-Learning (off-policy)', color='seagreen', linewidth=2, linestyle='--')
plt.title('SARSA vs Q-Learning — Tasa de éxito (media móvil 100 ep.)', fontsize=13)
plt.xlabel('Episodio')
plt.ylabel('Tasa de éxito (%)')
plt.ylim(0, 105)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Tasa de éxito final  — SARSA      : {sarsa_smooth[-1]:.1f}%')
print(f'Tasa de éxito final  — Q-Learning : {ql_smooth[-1]:.1f}%')

---
## Resumen

| Paso | ¿Qué hicimos? |
|------|---------------|
| 1–2  | Creamos el entorno FrozenLake con Gymnasium |
| 3    | Inicializamos la tabla Q a ceros |
| 4    | Definimos los hiperparámetros (α, γ, ε) |
| 5    | Implementamos la política ε-greedy |
| 6    | Implementamos la regla de actualización SARSA |
| 7    | Ejecutamos el bucle de entrenamiento completo |
| 8–10 | Inspeccionamos la tabla Q y la política aprendida |
| 11   | Evaluamos el agente entrenado sin exploración |
| 12   | Comparamos SARSA con Q-Learning |

### Conceptos clave

- **On-policy**: SARSA aprende de la política que realmente ejecuta (incluyendo la exploración)
- **Tabla Q**: estructura de datos central; no hay redes neuronales
- **TD error**: mide la diferencia entre lo esperado y lo recibido
- **ε-greedy**: mecanismo que equilibra exploración y explotación

### Siguientes pasos

- Probar con `is_slippery=True` para añadir estocasticidad
- Aumentar el número de episodios
- Experimentar con distintos valores de α y γ
- Aplicar el mismo esquema a otros entornos discretos (Taxi-v3, CliffWalking-v0)

In [ ]:
env.close()
print('Entorno cerrado correctamente.')